## Script Summary

This notebook builds a staged entity-normalization pipeline from curated Niemann-Pick (NPC) relationships, validates names with PrimeKG/UMLS, and maps validated entities back to relationship rows.

### Inputs
- Primary input table: `curated_relationships_df`
- Normalized relationship table: `novel_relationships_df`

### Main processing stages
1. Build initial query names from curated relationships.
2. Split names into exact PrimeKG matches and nonmatches.
3. Run UMLS normalization for nonmatched names.
4. Split first-search results into rows with CUI vs without CUI.
5. Run semantic-type quality check for first-search CUI rows.
6. Build second-round suggestion inputs.
7. Generate PrimeKG suggested replacements (SapBERT retrieve + SBERT rerank).
8. Re-check suggested names in UMLS and evaluate type consistency.
9. Build final valid-name pools across stages.
10. Map status/CUI back to curated relationship rows.
11. Add suggested-name mapping back to relationship rows.

### Key intermediate variables
- `query_names` (round 1 and round 2)
- `kg_matched_names`, `kg_nonmatched_names`
- `entity_umls_df`
- `entity_with_cui_after_first_search`, `entity_without_cui_after_first_search`
- `suggested_name_replacement_df`
- `suggested_umls_typecheck_df`
- `valid_names_after_first_search`, `valid_names_after_second_search`, `all_good_names`

### Final outputs used downstream
- `suggested_name_replacement_df`
- `suggested_umls_typecheck_df`
- `relationships_with_status_df`

In [ ]:
# --- Config & Imports ---
# Portable paths: override with env vars CURATION_ROOT, PLUS_KG, UMLS_API_KEY, RELEASE_ROOT.

import os
import pandas as pd
import re
import requests
import numpy as np
import torch
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from numpy.linalg import norm

_LIT_DIR = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
_RELEASE_ROOT = Path(os.environ.get("RELEASE_ROOT", str(_LIT_DIR.parents[1])))
CURATION_ROOT = Path(os.environ.get(
    "CURATION_ROOT",
    str(_RELEASE_ROOT / "dataset" / "PrimeKG-Plus-RD" / "curation_source"),
))
PLUS_KG = Path(os.environ.get("PLUS_KG", str(_RELEASE_ROOT / "dataset" / "PrimeKG-Plus" / "primekg_plus.csv")))
KG_FILE = PLUS_KG
POST_DIR = CURATION_ROOT / "Post curation"
BEFORE_BERT_DIR = POST_DIR / "before_bert"
FINALS_V1_DIR = POST_DIR / "finals_v1"
QC_OUTPUTS_DIR = POST_DIR / "qc_outputs"
INTERMEDIATE_DIR = POST_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
CURATED_CSV = CURATION_ROOT / "Pick Niemann disease/Final_Niemann-Pick Disease.csv"

UMLS_API_KEY = os.environ.get("UMLS_API_KEY", "")
if not UMLS_API_KEY:
    raise ValueError("Set UMLS_API_KEY (NLM UTS API key) before running UMLS search cells.")

SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
SAPBERT_MAX_LEN = 25
SAPBERT_BATCH = 64

TEST_MODE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
pwd

In [3]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|███████████████████████████████████████████████| 103/103 [00:00<00:00, 2054.41it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
ENTITY_TYPE_TO_TUI = {

    # Disease-related concepts
    "disease": [
        "T047",  # Disease or Syndrome (primary disease group)
        "T191",  # Neoplastic Process (tumor/cancer)
        "T048",  # Mental or Behavioral Dysfunction
    ],

    # Gene / protein
    "gene/protein": [
        "T028",  # Gene or Genome
        "T192",  # Receptor (can be treated as protein)
    ],

    # Gene / protein
    "protein/gene": [
        "T028",  # Gene or Genome
        "T192",  # Receptor (can be treated as protein)
    ],

    # Drug / chemical treatment
    "drug": [
        "T121",  # Pharmacologic Substance
        "T200",  # Clinical Drug
        "T195",  # Antibiotic (subset of drug)
    ],

    # Phenotype / symptom / abnormality
    "phenotype": [
        "T033",  # Finding (e.g., hypotonia, weakness)
        "T034",  # Laboratory or Test Result
        "T041",  # Mental Process
        "T184",  # Sign or Symptom
        "T019",  # Congenital Abnormality
        "T020",  # Acquired Abnormality
    ],

    # Biological process (broad)
    "biological_process": [
        "T038",  # Biologic Function
        "T039",  # Physiologic Function
    ],

    # Molecular function (GO-like)
    "molecular_function": [
        "T043",  # Cell Function
    ],

    # Cellular component
    "cellular_component": [
        "",
    ],

    # Pathway (no exact TUI; closest approximation)
    "pathway": [
        "T038",  # Biologic Function
    ],

    # Anatomy
    "anatomy": [
        "T017",  # Anatomical Structure
        "T018",  # Embryonic Structure
        "T021",  # Fully Formed Anatomical Structure
        "T022",  # Body System
        "T023",  # Body Part, Organ, or Organ Component
        "T024",  # Tissue
        "T029",  # Body Location or Region
        "T030",  # Body Space or Junction
    ],

    # Exposure / procedure / measurement
    "exposure": [
        "T060",  # Diagnostic Procedure (e.g., MRI, ultrasound)
        "T061",  # Therapeutic or Preventive Procedure
        "T063",  # Molecular Biology Research Technique
    ],

    # Pathological process
    "pathology": [
        "T046",  # Pathologic Function (e.g., inflammation)
    ],
}

In [5]:
# Load curated relationships
if not CURATED_CSV.exists():
    raise FileNotFoundError(f"Curated CSV not found: {CURATED_CSV}")
curated_relationships_df = pd.read_csv(CURATED_CSV)
curated_relationships_df.columns

Index(['STT', 'doi/PMID', 'Abstract/Introduction', 'Journal type',
       'experiment', 'model', 'relation ', 'display_relation', 'x_name ',
       'x_type ', 'y_name', 'y_type', 'note', 'expert opinion', 'Unnamed: 14',
       'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18',
       'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22',
       'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26',
       'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30',
       'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34',
       'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38',
       'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42',
       'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46',
       'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50',
       'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54'],
      dtype='str')

In [6]:
# Standardize known column names
curated_relationships_df = curated_relationships_df[['doi/PMID', 'Abstract/Introduction', 'Journal type',
       'experiment', 'model', 'relation ', 'display_relation', 'x_name ',
       'x_type ', 'y_name', 'y_type', 'note', 'expert opinion']]
curated_relationships_df.columns = ['PMID', 'ABIN', 'Journal type', 'experiment',
       'model', 'relation', 'display_relation', 'x_name', 'x_type',
       'y_name', 'y_type', 'note', 'expert opinion']
curated_relationships_df = curated_relationships_df.drop(columns=['display_relation', 'note', 'expert opinion'])
curated_relationships_df.head(3)

,PMID,ABIN,Journal type,experiment,model,relation,x_name,x_type,y_name,y_type
0,34364974,Abstract,original article,in vivo,"human, mice",disease_protein,Niemann-Pick type C disease,disease,NPC1,protein/gene
1,34364974,Abstract,original article,in vivo,"human, mice",disease_protein,Niemann-Pick type C disease,disease,NPC2,protein/gene
2,34364974,Abstract,original article,in vivo,"human, mice",indication,2-hydroxypropyl-beta-cyclodextrin,drug,Niemann-Pick Disease Type C,disease


In [7]:
len(curated_relationships_df)

5824

In [8]:
curated_relationships_df.tail(3)

,PMID,ABIN,Journal type,experiment,model,relation,x_name,x_type,y_name,y_type
5821,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5822,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5823,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
curated_relationships_df = curated_relationships_df.dropna()
len(curated_relationships_df)

2939

In [10]:
relationship_rows = []
for _, row in curated_relationships_df.iterrows():
    entity1_name = row.get('x_name')
    entity2_name = row.get('y_name')
    if not (isinstance(entity1_name, str) and entity1_name.strip() and isinstance(entity2_name, str) and entity2_name.strip()):
        continue
    relationship_rows.append({
        'entity1': entity1_name.strip(),
        'entity2': entity2_name.strip(),
        'entity_type1': row.get('x_type'),
        'entity_type2': row.get('y_type'),
        'Relation': row.get('relation'),
        'PMID': row.get('PMID'),
    })

novel_relationships_df = pd.DataFrame(relationship_rows)
novel_relationships_df.head(3)


,entity1,entity2,entity_type1,entity_type2,Relation,PMID
0,Niemann-Pick type C disease,NPC1,disease,protein/gene,disease_protein,34364974
1,Niemann-Pick type C disease,NPC2,disease,protein/gene,disease_protein,34364974
2,2-hydroxypropyl-beta-cyclodextrin,Niemann-Pick Disease Type C,drug,disease,indication,34364974


In [11]:
len(novel_relationships_df)

2939

In [12]:
# String exact match against PrimeKG names
# Output lists required by next steps:
# - kg_matched_names
# - kg_nonmatched_names
import re
import unicodedata
def _norm_name(x: str) -> str:
    s = unicodedata.normalize("NFKC", str(x))
    s = s.strip().lower()
    s = s.replace("‑", "-").replace("–", "-").replace("—", "-")
    s = re.sub(r"\s+", " ", s)
    return s

_norm_name("Canavan   disease ")

'canavan disease'

In [13]:
query_names = []
if "query_names" not in globals() or query_names is None or len(query_names) == 0:
    q1 = novel_relationships_df.get("entity1", pd.Series(dtype=str)).dropna().astype(str)
    q2 = novel_relationships_df.get("entity2", pd.Series(dtype=str)).dropna().astype(str)
    query_names = sorted({
        _norm_name(x)
        for x in pd.concat([q1, q2], ignore_index=True).tolist()
        if _norm_name(x) and _norm_name(x) != "nan"
    })
len(query_names), query_names[:3]

(2029,
 [', improved gross motor skills',
  '11β-hydroxysteroid-dehydrogenase of type 2',
  '11β-hydroxysteroid-dehydrogenase of type 2 inhibitor'])

In [14]:
# Build PrimeKG name pool if not already prepared
primekg_entity_names_list = []
if "primekg_entity_names_list" not in globals() or primekg_entity_names_list is None or len(primekg_entity_names_list) == 0:
    kg_df = pd.read_csv(KG_FILE, low_memory=False)
    name_cols = [c for c in kg_df.columns if c.endswith("_name")]
    if not name_cols:
        raise ValueError(f"No columns ending with _name in {KG_FILE}")
    acc = []
    for c in name_cols:
        acc.extend(kg_df[c].dropna().astype(str).tolist())
    # normalize first, then deduplicate
    primekg_entity_names_list = sorted({
        _norm_name(x)
        for x in acc
        if _norm_name(x) and _norm_name(x) != "nan"
    })
primekg_name_norm_set = set(primekg_entity_names_list)
len(primekg_name_norm_set)

127561

In [15]:
kg_matched_names = []
kg_nonmatched_names = []
for q in query_names:
    q_clean = str(q).strip()
    if not q_clean or q_clean.lower() == "nan":
        continue
    if _norm_name(q_clean) in primekg_name_norm_set:
        kg_matched_names.append(q_clean)
    else:
        kg_nonmatched_names.append(q_clean)

In [16]:
# Unique + sorted for deterministic downstream behavior
kg_matched_names = sorted(set(kg_matched_names))
kg_nonmatched_names = sorted(set(kg_nonmatched_names))

# Build dictionaries requested: name -> curated entity_type(s)
name_to_entity_types = {}
for _, r in novel_relationships_df.iterrows():
    n1 = str(r.get("entity1") or "").strip()
    t1 = str(r.get("entity_type1") or "").strip()
    if n1 and n1.lower() != "nan" and t1 and t1.lower() != "nan":
        name_to_entity_types.setdefault(n1, set()).add(t1)

    n2 = str(r.get("entity2") or "").strip()
    t2 = str(r.get("entity_type2") or "").strip()
    if n2 and n2.lower() != "nan" and t2 and t2.lower() != "nan":
        name_to_entity_types.setdefault(n2, set()).add(t2)

In [17]:
name_to_entity_types

{'Niemann-Pick type C disease': {'disease'},
 'NPC1': {'disease', 'gene/protein', 'phenotype', 'protein/gene'},
 'NPC2': {'anatomy', 'gene/protein', 'protein/gene'},
 '2-hydroxypropyl-beta-cyclodextrin': {'drug'},
 'Niemann-Pick Disease Type C': {'disease', 'drug', 'protein/gene'},
 'neurovisceral symptoms': {'phenotype'},
 'accumulation of cholesterol and other lipids': {'phenotype'},
 'release of lipid-laden lamellar inclusions': {'biological_process'},
 'restore lipid homeostasis': {'biological_process'},
 'Niemann-Pick Disease Type C2': {'disease'},
 'fatal neurovisceral disorder': {'phenotype'},
 'splenomegaly': {'phenotype'},
 'cerebellar ataxia': {'disease', 'phenotype'},
 'growth retardation': {'phenotype'},
 'tremor': {'phenotype'},
 'brain atrophy': {'phenotype'},
 'neuroinflammation': {'biological_process', 'pathology', 'phenotype'},
 'foam cell accumulation': {'phenotype'},
 'lungs': {'anatomy'},
 'liver': {'anatomy'},
 'spleen': {'anatomy'},
 'cerebellum': {'anatomy'},
 'l

In [18]:
kg_matched_entities = {
    n: " | ".join(sorted(name_to_entity_types.get(n, set())))
    for n in kg_matched_names
}
kg_nonmatched_entities = {
    n: " | ".join(sorted(name_to_entity_types.get(n, set())))
    for n in kg_nonmatched_names
}

print(f"query_names total: {len(query_names)}")
print(f"kg_matched_names: {len(kg_matched_names)}")
print(f"kg_nonmatched_names: {len(kg_nonmatched_names)}")

routing_preview_df = pd.DataFrame(
    {
        "bucket": ["kg_matched_names", "kg_nonmatched_names"],
        "count": [len(kg_matched_names), len(kg_nonmatched_names)],
    }
)
routing_preview_df

query_names total: 2029
kg_matched_names: 246
kg_nonmatched_names: 1783


,bucket,count
0,kg_matched_names,246
1,kg_nonmatched_names,1783


In [19]:
# UMLS search helpers (restored)
UMLS_BASE = 'https://uts-ws.nlm.nih.gov/rest'


def _umls_search_best(term, api_key, max_results=10):
    """Return first valid CUI match from UMLS search (no type constraint)."""
    try:
        import re

        url = f"{UMLS_BASE}/search/current"
        params = {
            'apiKey': api_key,
            'string': term,
            'pageSize': max_results,
            'searchType': 'words',
        }
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()

        results = r.json().get('result', {}).get('results', [])
        for x in results:
            ui = (x.get('ui') or '').strip()
            if re.fullmatch(r'C\d+', ui):
                return ui, x.get('name', '')
        return None
    except Exception:
        return None


def _umls_get_semantic_types(cui, api_key):
    """Get semantic type codes (TUI) + names; includes fallback endpoint."""
    import re

    def _extract(st_list):
        tuis, names = [], []
        for st in st_list or []:
            if isinstance(st, dict):
                code = (
                    st.get('uri')
                    or st.get('tui')
                    or st.get('ui')
                    or st.get('code')
                    or st.get('name')
                )
                nm = st.get('name', '')
            else:
                code = str(st)
                nm = ''
            code = str(code)
            m = re.search(r'T\d{3}', code)
            if m:
                tuis.append(m.group(0))
                names.append(nm)
        return tuis, names

    try:
        # Primary endpoint
        r = requests.get(
            f"{UMLS_BASE}/content/current/CUI/{cui}",
            params={'apiKey': api_key},
            timeout=30,
        )
        r.raise_for_status()
        tuis, names = _extract(r.json().get('result', {}).get('semanticTypes', []))
        if tuis or names:
            return tuis, names

        # Fallback endpoint
        r2 = requests.get(
            f"{UMLS_BASE}/content/current/CUI/{cui}/semantictypes",
            params={'apiKey': api_key},
            timeout=30,
        )
        r2.raise_for_status()
        res = r2.json().get('result', [])
        st_list = res.get('semanticTypes', []) if isinstance(res, dict) else res
        return _extract(st_list)
    except Exception:
        return [], []

In [20]:
umls_rows = []
for name in tqdm(kg_nonmatched_names, desc='UMLS normalization'):
    search_term = _norm_name(name)
    original_entity_types =  kg_nonmatched_entities.get(name)
    best = _umls_search_best(search_term, UMLS_API_KEY)
    if not best:
        umls_rows.append({
            'entity_name': name,
            'original_entity_types': original_entity_types,
            'search_term_used': search_term,
            'umls_matched_name': None,
            'matched_cui': None,
            'matched_semantic_types': [],
            'matched_semantic_type_names': []
        })
        continue
    cui, umls_name = best
    st, st_names = _umls_get_semantic_types(cui, UMLS_API_KEY)
    umls_rows.append({
        'entity_name': name,
        'original_entity_types': original_entity_types,
        'search_term_used': search_term,
        'umls_matched_name': umls_name,
        'matched_cui': cui,
        'matched_semantic_types': st,
        'matched_semantic_type_names': st_names
    })

entity_umls_df = pd.DataFrame(umls_rows)


UMLS normalization: 100%|█████████████████████████████████████████████████████████████████████████████████████| 1783/1783 [20:35<00:00,  1.44it/s]


In [51]:
len(novel_relationships_df)

2939

In [52]:
# Add PMID(s), relation(s), and entity_side(s) for each entity in entity_umls_df
name_to_pmids = {}
name_to_relations = {}
name_to_entity_sides = {}

count=0
for _, r in novel_relationships_df.iterrows():
    print(count)
    count=count+1
    pmid = str(r.get("PMID") or "").strip()
    relation = str(r.get("Relation") or "").strip()

    n1 = str(r.get("entity1") or "").strip()
    if n1 and n1.lower() != "nan":
        if pmid and pmid.lower() != "nan":
            name_to_pmids.setdefault(n1, set()).add(pmid)
        if relation and relation.lower() != "nan":
            name_to_relations.setdefault(n1, set()).add(relation)
        name_to_entity_sides.setdefault(n1, set()).add("1")

    n2 = str(r.get("entity2") or "").strip()
    if n2 and n2.lower() != "nan":
        if pmid and pmid.lower() != "nan":
            name_to_pmids.setdefault(n2, set()).add(pmid)
        if relation and relation.lower() != "nan":
            name_to_relations.setdefault(n2, set()).add(relation)
        name_to_entity_sides.setdefault(n2, set()).add("2")

entity_umls_df["PMID"] = entity_umls_df["entity_name"].map(
    lambda x: "|".join(sorted(name_to_pmids.get(str(x).strip(), set())))
)
entity_umls_df["relation"] = entity_umls_df["entity_name"].map(
    lambda x: "|".join(sorted(name_to_relations.get(str(x).strip(), set())))
)
entity_umls_df["entity_side"] = entity_umls_df["entity_name"].map(
    lambda x: "|".join(sorted(name_to_entity_sides.get(str(x).strip(), set())))
)
# Cleanup typo column from previous runs if it exists.
if "entitiy_side" in entity_umls_df.columns:
    entity_umls_df = entity_umls_df.drop(columns=["entitiy_side"])
entity_umls_df.head(5)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

,entity_name,original_entity_types,search_term_used,umls_matched_name,matched_cui,matched_semantic_types,matched_semantic_type_names,PMID,relation,entity_side
1,11β-hydroxysteroid-dehydrogenase of type 2,protein/gene,11β-hydroxysteroid-dehydrogenase of type 2,NaN,NaN,[],[],37683773,phenotype_protein,2
2,11β-hydroxysteroid-dehydrogenase of type 2 inh...,biological_process,11β-hydroxysteroid-dehydrogenase of type 2 inh...,NaN,NaN,[],[],37683773,chemical_bioprocess,2
3,2-hydroxypropyl-b-cyclodextrin,drug,2-hydroxypropyl-b-cyclodextrin,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",35081253|36356931|38448301,drug_bioprocess|drug_effect|indication,1
4,2-hydroxypropyl-beta-cyclodextrin,drug,2-hydroxypropyl-beta-cyclodextrin,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",34150357|34364974|35452076|36431844|36920149|3...,drug_bioprocess|drug_drug|drug_effect|indicati...,1|2
5,2-hydroxypropyl-beta-cyclodextrin.,drug,2-hydroxypropyl-beta-cyclodextrin.,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",36431844,anatomy_protein_present|drug_effect,1|2


In [53]:
#dont change code above this line
entity_umls_df_bkup = entity_umls_df.copy()
entity_umls_df = entity_umls_df_bkup
entity_umls_df.head(10)

,entity_name,original_entity_types,search_term_used,umls_matched_name,matched_cui,matched_semantic_types,matched_semantic_type_names,PMID,relation,entity_side
1,11β-hydroxysteroid-dehydrogenase of type 2,protein/gene,11β-hydroxysteroid-dehydrogenase of type 2,NaN,NaN,[],[],37683773,phenotype_protein,2
2,11β-hydroxysteroid-dehydrogenase of type 2 inh...,biological_process,11β-hydroxysteroid-dehydrogenase of type 2 inh...,NaN,NaN,[],[],37683773,chemical_bioprocess,2
3,2-hydroxypropyl-b-cyclodextrin,drug,2-hydroxypropyl-b-cyclodextrin,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",35081253|36356931|38448301,drug_bioprocess|drug_effect|indication,1
4,2-hydroxypropyl-beta-cyclodextrin,drug,2-hydroxypropyl-beta-cyclodextrin,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",34150357|34364974|35452076|36431844|36920149|3...,drug_bioprocess|drug_drug|drug_effect|indicati...,1|2
5,2-hydroxypropyl-beta-cyclodextrin.,drug,2-hydroxypropyl-beta-cyclodextrin.,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",36431844,anatomy_protein_present|drug_effect,1|2
6,2-hydroxypropyl-ß-cyclodextrin,drug,2-hydroxypropyl-ß-cyclodextrin,NaN,NaN,[],[],37210540,indication,1
7,2-hydroxypropyl-β-cyclodextrin,disease | drug,2-hydroxypropyl-β-cyclodextrin,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",34296265|34592985|35140266|35891519|36116252|3...,disease_disease|drug_bioprocess|drug_drug|drug...,1
10,2-hydroxypropyl-γ-cyclodextrin,drug,2-hydroxypropyl-γ-cyclodextrin,NaN,NaN,[],[],35891519|38852794,drug_bioprocess|drug_drug|drug_effect|indication,1
12,"3β,5α,6β-trihydroxycholanoyl-glycine",phenotype,"3β,5α,6β-trihydroxycholanoyl-glycine",NaN,NaN,[],[],35833379,disease_phenotype_positive,2
13,4-sulfobutylether-β-cyclodextrin,drug,4-sulfobutylether-β-cyclodextrin,NaN,NaN,[],[],36584173,drug_effect,1


In [54]:
# Add semantic-type quality flag: out_of_expected_tuis (only meaningful when CUI exists)
import ast

def _to_list_safe(x):
    if isinstance(x, list):
        return x
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, list):
                return v
        except Exception:
            pass
    return [s]

#test
# Case 1: Already a list — keep as-is
print(_to_list_safe(["T047", "T048"]))         # ["T047", "T048"]

# Case 2: None → []
print(_to_list_safe(None))                      # []

# Case 3: NaN → []
print(_to_list_safe(float("nan")))              # []
print(_to_list_safe(pd.NA))                     # []

# Case 4: Empty string → []
print(_to_list_safe("   "))                     # []

# Case 5: Valid list string → parse to list
print(_to_list_safe("['T047', 'T048']"))        # ["T047", "T048"]
print(_to_list_safe("['T047']"))                # ["T047"]

# Case 6: Invalid list string → wrap as [s]
print(_to_list_safe("[abc, def]"))              # ["[abc, def]"]

# Case 7: Plain string → wrap as list
print(_to_list_safe("T047"))                    # ["T047"]
print(_to_list_safe("  Canavan disease  "))     # ["Canavan disease"]

['T047', 'T048']
[]
[]
['<NA>']
[]
['T047', 'T048']
['T047']
['[abc, def]']
['T047']
['Canavan disease']


In [55]:
import ast

def _to_list_safe(x):
    if isinstance(x, list):
        return x
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, list):
                return v
        except Exception:
            pass
    return [s]


def _expected_tuis_from_entity_types(entity_types):
    exp = set()
    for et in _to_list_safe(entity_types):
        key = str(et).strip().lower()
        for tui in ENTITY_TYPE_TO_TUI.get(key, []):
            tui_clean = str(tui).strip().upper()
            if tui_clean:
                exp.add(tui_clean)
    return exp


def _observed_tuis(semantic_types_col):
    obs = set()
    for t in _to_list_safe(semantic_types_col):
        tv = str(t).strip().upper()
        if tv:
            obs.add(tv)
    return obs



In [56]:
for t in ["disease", "phenotype", "drug"]:
    print(_expected_tuis_from_entity_types(t))

{'T191', 'T048', 'T047'}
{'T041', 'T020', 'T033', 'T034', 'T184', 'T019'}
{'T200', 'T195', 'T121'}


In [57]:
print(len(entity_umls_df))
entity_umls_df = entity_umls_df[entity_umls_df.original_entity_types!=""]
print(len(entity_umls_df))


1131
1131


In [58]:
entity_with_cui_after_first_search = entity_umls_df[~entity_umls_df.matched_cui.isna()]
len(entity_with_cui_after_first_search)

535

In [59]:
entity_umls_df.columns

Index(['entity_name', 'original_entity_types', 'search_term_used',
       'umls_matched_name', 'matched_cui', 'matched_semantic_types',
       'matched_semantic_type_names', 'PMID', 'relation', 'entity_side'],
      dtype='str')

In [60]:
entity_without_cui_after_first_search = entity_umls_df[entity_umls_df.matched_cui.isna()]
entity_without_cui_after_first_search = entity_without_cui_after_first_search[['entity_name', 'original_entity_types', "PMID", 'relation', 'entity_side']]
entity_without_cui_after_first_search

,entity_name,original_entity_types,PMID,relation,entity_side
1,11β-hydroxysteroid-dehydrogenase of type 2,protein/gene,37683773,phenotype_protein,2
2,11β-hydroxysteroid-dehydrogenase of type 2 inh...,biological_process,37683773,chemical_bioprocess,2
6,2-hydroxypropyl-ß-cyclodextrin,drug,37210540,indication,1
10,2-hydroxypropyl-γ-cyclodextrin,drug,35891519|38852794,drug_bioprocess|drug_drug|drug_effect|indication,1
12,"3β,5α,6β-trihydroxycholanoyl-glycine",phenotype,35833379,disease_phenotype_positive,2
...,...,...,...,...,...
1762,visceral manifestations,phenotype,36325261,disease_phenotype_positive,2
1763,visceral organ damage,phenotype,34061502,disease_phenotype_positive,2
1771,weakened muscular contraction,phenotype,36064725,disease_phenotype_positive,2
1772,widespread neurodegeneration,phenotype,39202426,phenotype_protein,2


In [61]:
entity_without_cui_after_first_search.to_csv(str(POST_DIR / "2026058-NPC_disease_For_term_suggestion_before_BERT.csv"))

In [62]:
out_flags = []
expected_tuis_col = []

for _, r in entity_with_cui_after_first_search.iterrows(): #only apply for entity that has cui after searching
    cui = r.get("matched_cui")
    has_cui = not (
        cui is None
        or (isinstance(cui, float) and pd.isna(cui))
        or str(cui).strip() == ""
    )
    print("*** \n ",cui, " has cui ", has_cui)
    
    expected_tuis = _expected_tuis_from_entity_types(r.get("original_entity_types", []))
    observed_tuis = _observed_tuis(r.get("matched_semantic_types", []))  # use correct column
    
    expected_tuis_col.append(sorted(expected_tuis))
    
    # out_of_expected_tuis = True khi:
    # - has CUI (comparison meaningful)
    # - has both expected and observed
    # - no overlap between observed and expected
    print("expected_tuis: ", expected_tuis, "observed_tuis: ", observed_tuis), 
    out_of_expected = bool(
        has_cui
        and bool(expected_tuis)
        and bool(observed_tuis)
        and len(expected_tuis & observed_tuis) == 0  # ← changed from > 0 to == 0
    )
    print(len(expected_tuis & observed_tuis), len(expected_tuis & observed_tuis)==0)
    print("out_of_expected: ", out_of_expected)
    out_flags.append(out_of_expected)

entity_with_cui_after_first_search["expected_tuis"] = expected_tuis_col
entity_with_cui_after_first_search["out_of_expected_tuis"] = pd.Series(
    out_flags,
    index=entity_with_cui_after_first_search.index,
    dtype=bool,
)
# Defensive cleanup in case this cell is re-run after earlier buggy assignments.
entity_with_cui_after_first_search["out_of_expected_tuis"] = (
    entity_with_cui_after_first_search["out_of_expected_tuis"]
    .fillna(False)
    .astype(bool)
)

*** 
  C0046237  has cui  True
expected_tuis:  {'T200', 'T195', 'T121'} observed_tuis:  {'T121', 'T109'}
1 False
out_of_expected:  False
*** 
  C0046237  has cui  True
expected_tuis:  {'T200', 'T195', 'T121'} observed_tuis:  {'T121', 'T109'}
1 False
out_of_expected:  False
*** 
  C0046237  has cui  True
expected_tuis:  {'T200', 'T195', 'T121'} observed_tuis:  {'T121', 'T109'}
1 False
out_of_expected:  False
*** 
  C0046237  has cui  True
expected_tuis:  set() observed_tuis:  {'T121', 'T109'}
0 True
out_of_expected:  False
*** 
  C0049920  has cui  True
expected_tuis:  set() observed_tuis:  {'T109'}
0 True
out_of_expected:  False
*** 
  C5980104  has cui  True
expected_tuis:  {'T041', 'T020', 'T033', 'T034', 'T184', 'T019'} observed_tuis:  {'T033'}
1 False
out_of_expected:  False
*** 
  C2607886  has cui  True
expected_tuis:  set() observed_tuis:  {'T121', 'T109'}
0 True
out_of_expected:  False
*** 
  C5927802  has cui  True
expected_tuis:  {'T041', 'T020', 'T033', 'T034', 'T184', 'T019

In [63]:
entity_with_cui_after_first_search

,entity_name,original_entity_types,search_term_used,umls_matched_name,matched_cui,matched_semantic_types,matched_semantic_type_names,PMID,relation,entity_side,expected_tuis,out_of_expected_tuis
3,2-hydroxypropyl-b-cyclodextrin,drug,2-hydroxypropyl-b-cyclodextrin,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",35081253|36356931|38448301,drug_bioprocess|drug_effect|indication,1,"[T121, T195, T200]",False
4,2-hydroxypropyl-beta-cyclodextrin,drug,2-hydroxypropyl-beta-cyclodextrin,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",34150357|34364974|35452076|36431844|36920149|3...,drug_bioprocess|drug_drug|drug_effect|indicati...,1|2,"[T121, T195, T200]",False
5,2-hydroxypropyl-beta-cyclodextrin.,drug,2-hydroxypropyl-beta-cyclodextrin.,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",36431844,anatomy_protein_present|drug_effect,1|2,"[T121, T195, T200]",False
7,2-hydroxypropyl-β-cyclodextrin,disease | drug,2-hydroxypropyl-β-cyclodextrin,hydroxypropylbetadex (0.58-0.68 MS),C0046237,"[T109, T121]","[Organic Chemical, Pharmacologic Substance]",34296265|34592985|35140266|35891519|36116252|3...,disease_disease|drug_bioprocess|drug_drug|drug...,1,[],False
17,7-ketocholesterol,phenotype | protein/gene,7-ketocholesterol,7-ketocholesterol,C0049920,[T109],[Organic Chemical],34303826|38131230,disease_protein|phenotype_phenotype|phenotype_...,1|2,[],False
...,...,...,...,...,...,...,...,...,...,...,...,...
1767,visceral symptoms,phenotype,visceral symptoms,visceral; symptoms purpura,C1405869,[T033],[Finding],34303826|34304992|35487270|38087834,disease_phenotype_positive,2,"[T019, T020, T033, T034, T041, T184]",False
1768,voltage-gated calcium channels,cellular_component,voltage-gated calcium channels,clustering of voltage-gated calcium channels,C2610722,[T043],[Cell Function],37507375,cellcomp_cellcomp|protein_protein,2,[],False
1770,vulnerability to ferroptosis,phenotype,vulnerability to ferroptosis,LINC01770 gene,C4320850,[T028],[Gene or Genome],40243519,phenotype_protein,2,"[T019, T020, T033, T034, T041, T184]",True
1775,worry,phenotype,worry,Worried,C0233481,[T033],[Finding],34819124,disease_phenotype_positive,2,"[T019, T020, T033, T034, T041, T184]",False


In [64]:
# Threshold: entity is considered "in KG" by SapBERT if max cosine sim with KG >= threshold
SAPBERT_IN_KG_THRESHOLD = 0.5
# Top-k candidates from SapBERT to rerank with SBERT
TOP_K_SAPBERT = 20

In [65]:
# --- SapBERT: load model and define embed/cosine helpers ---
# Input: SAPBERT_MODEL, DEVICE (from config).
# Expected output: tokenizer, sapbert_model; functions embed_names, cos_sim.

def load_sapbert():
    tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
    model = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE)
    return tokenizer, model

def embed_names(tokenizer, model, names, batch_size=None):
    if not names:
        return np.zeros((0, 768))
    batch_size = batch_size or SAPBERT_BATCH
    names = [str(n).strip() or " " for n in names]
    embs = []
    for i in range(0, len(names), batch_size):
        batch = names[i : i + batch_size]
        toks = tokenizer(batch, padding="max_length", max_length=SAPBERT_MAX_LEN, truncation=True, return_tensors="pt")
        toks = {k: v.to(DEVICE) for k, v in toks.items()}
        with torch.no_grad():
            cls_rep = model(**toks)[0][:, 0, :]
        embs.append(cls_rep.cpu().numpy())
    return np.concatenate(embs, axis=0)

def cos_sim(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-9))

tokenizer, sapbert_model = load_sapbert()

Loading weights: 100%|███████████████████████████████████████████████| 199/199 [00:00<00:00, 2019.71it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [66]:
def load_sbert():
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
    return model

def embed_names_sbert(model, names):
    return model.encode(
        names,
        convert_to_numpy=True,
        normalize_embeddings=True  # important for cosine via dot product
    )

sbert_model = load_sbert()

Loading weights: 100%|███████████████████████████████████████████████| 103/103 [00:00<00:00, 2267.82it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [67]:
# Cell 107: SapBERT top-20 nearest PrimeKG names, then SBERT rerank
# Requires: novel_relationships_df (with entity*_cui, entity*_tui_out_of_expected_group),
#           tokenizer, sapbert_model, embed_names, DEVICE, SAPBERT_BATCH,
#           sbert_model, embed_names_sbert, KG_FILE
TOP_K = 20

def load_primekg_display_names(kg_path: Path) -> list[str]:
    kg = pd.read_csv(kg_path, low_memory=False)
    name_cols = [c for c in kg.columns if c.endswith("_name")]
    if not name_cols:
        raise ValueError(f"No columns ending with _name in {kg_path}")
    acc: list[str] = []
    for c in name_cols:
        acc.extend(kg[c].dropna().astype(str).str.strip().tolist())
    return sorted({x for x in acc if x and x.lower() != "nan"})


def _l2norm_rows(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-9)

def _normalize_curated_pmid(x) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    s = str(x).strip()
    if not s or s.lower() == "nan":
        return ""
    if s.endswith(".0") and s[:-2].replace(".", "", 1).isdigit():
        s = s[:-2]
    return s


primekg_entity_names_list = load_primekg_display_names(KG_FILE)

In [68]:
len(primekg_entity_names_list)

128278

In [69]:
len(entity_with_cui_after_first_search), len(entity_without_cui_after_first_search)

(535, 596)

In [70]:
len(entity_with_cui_after_first_search[~entity_with_cui_after_first_search.out_of_expected_tuis])

242

In [71]:
entity_with_cui_after_first_search.columns, entity_without_cui_after_first_search.columns

(Index(['entity_name', 'original_entity_types', 'search_term_used',
        'umls_matched_name', 'matched_cui', 'matched_semantic_types',
        'matched_semantic_type_names', 'PMID', 'relation', 'entity_side',
        'expected_tuis', 'out_of_expected_tuis'],
       dtype='str'),
 Index(['entity_name', 'original_entity_types', 'PMID', 'relation',
        'entity_side'],
       dtype='str'))

In [72]:
query_names2 = list(entity_with_cui_after_first_search[~entity_with_cui_after_first_search.out_of_expected_tuis].entity_name.unique()) + list(entity_without_cui_after_first_search.entity_name.unique())
query_names2 = list(set(query_names2))
# primekg_entity_names_list = load_primekg_display_names(KG_FILE)
len(query_names2)

838

In [73]:
# --- SapBERT: embed all KG names and all query names ---
#time consuming, run once
# kg_sapbert_embeddings = embed_names(tokenizer, sapbert_model, primekg_entity_names_list)

In [74]:
query_sapbert_embeddings = embed_names(tokenizer, sapbert_model, query_names2)
query_sbert_embeddings = embed_names_sbert(sbert_model, query_names2)

In [75]:
#load heavy embeddings from files
EMBED_DIR = "./embeddings_cache"

kg_sapbert_embeddings = np.load(f"{EMBED_DIR}/kg_sapbert_embeddings.npy")
kg_sbert_embeddings   = np.load(f"{EMBED_DIR}/kg_sbert_embeddings.npy")

with open(f"{EMBED_DIR}/primekg_entity_names_list.txt") as f:
    primekg_entity_names_list = f.read().splitlines()

print(f"Loaded SapBERT: {kg_sapbert_embeddings.shape}")
print(f"Loaded SBERT:   {kg_sbert_embeddings.shape}")

Loaded SapBERT: (128278, 768)
Loaded SBERT:   (128278, 384)


In [76]:
kg_sap_n = _l2norm_rows(kg_sapbert_embeddings)
q_sap_n = _l2norm_rows(query_sapbert_embeddings)


k = min(TOP_K, len(primekg_entity_names_list))

In [77]:
import os
# os.makedirs(EMBED_DIR, exist_ok=True)

# np.save(f"{EMBED_DIR}/kg_sapbert_embeddings.npy", kg_sapbert_embeddings)
# np.save(f"{EMBED_DIR}/kg_sbert_embeddings.npy",   kg_sbert_embeddings)

# # Save name list for later verification
# with open(f"{EMBED_DIR}/primekg_entity_names_list.txt", "w") as f:
#     f.write("\n".join(primekg_entity_names_list))

# print(f"Saved {kg_sapbert_embeddings.shape} SapBERT embeddings")
# print(f"Saved {kg_sbert_embeddings.shape} SBERT embeddings")

In [80]:
rows: list[dict] = []

for i, orig in enumerate(query_names2):
    sims = kg_sap_n @ q_sap_n[i]  # (n_kg,)
    if k >= len(sims):
        top_idx = np.argsort(-sims)
    else:
        # argpartition for top-k without full sort
        part = np.argpartition(-sims, k - 1)[:k]
        top_idx = part[np.argsort(-sims[part])]

    top_idx = top_idx[:k]
    qvec = query_sbert_embeddings[i : i + 1]  # (1, dim)
    sbert_scores = np.dot(kg_sbert_embeddings[top_idx], qvec.T).flatten()
    best_rel = int(np.argmax(sbert_scores))
    best_kg_idx = int(top_idx[best_rel])
    suggested = primekg_entity_names_list[best_kg_idx]
    rows.append({"original_name": orig, "suggested_name": suggested})

suggested_name_replacement_df = pd.DataFrame(rows, columns=["original_name", "suggested_name"])

# Enrich with metadata columns requested by downstream export.
extra_cols = ["entity_name", "original_entity_types", "PMID", "relation", "entity_side"]
meta_df = entity_umls_df[extra_cols].drop_duplicates(subset=["entity_name"], keep="first")
meta_df = meta_df.rename(columns={"entity_name": "original_name"})
suggested_name_replacement_df = suggested_name_replacement_df.merge(
    meta_df,
    on="original_name",
    how="left",
)

suggested_name_replacement_df = suggested_name_replacement_df[[
    "original_name",
    "suggested_name",
    "original_entity_types",
    "PMID",
    "relation",
    "entity_side",
]]

# suggested_name_replacement_df.head(15)

In [81]:
len(suggested_name_replacement_df)

838

In [82]:
ENTITY_TYPE_ALIASES = {"protein/gene": "gene/protein"}

def _etype_to_expected_tuis(etype):
    if pd.isna(etype) or etype == "":
        return set()
    key = str(etype).strip().lower()
    key = ENTITY_TYPE_ALIASES.get(key, key)
    return set(tui.upper() for tui in (ENTITY_TYPE_TO_TUI.get(key, []) or []))


In [83]:
def _expected_type_label_and_tuis_for_original_name(orig: str) -> tuple[str, set[str]]:
    """From suggestion metadata for original name, build union of expected TUIs."""
    o = str(orig).strip()
    matched_types = suggested_name_replacement_df.loc[
        suggested_name_replacement_df["original_name"] == o,
        "original_entity_types",
    ].dropna()
    if matched_types.empty:
        return "", set()

    raw = str(matched_types.iloc[0]).strip()
    labels = sorted({x.strip() for x in raw.split("|") if x.strip()})
    if not labels:
        return "", set()
    exp_union: set[str] = set()
    for lab in labels:
        exp_union |= _etype_to_expected_tuis(lab)
    exp_union.discard("")  # skip empty TUI from ENTITY_TYPE_TO_TUI mapping
    return " | ".join(labels), exp_union


def build_suggested_umls_typecheck_df(suggestions: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict] = []
    for _, r in tqdm(suggestions.iterrows(), total=len(suggestions), desc="UMLS re-search suggested names"):
        orig = r["original_name"]
        sug = r["suggested_name"]
        pmid = r.get("PMID", pd.NA)
        relation = r.get("relation", pd.NA)
        entity_side = r.get("entity_side", pd.NA)
        exp_label, exp_tuis = _expected_type_label_and_tuis_for_original_name(orig)
        # print(orig, "|", sug, "|", exp_label, "|", exp_tuis )

        best = _umls_search_best(str(sug).strip(), UMLS_API_KEY)
        if not best:
            rows.append(
                {
                    "original_name": orig,
                    "suggested_name": sug,
                    "expected_entity_type": exp_label,
                    "expected_tuis": ",".join(sorted(exp_tuis)),
                    "PMID": pmid,
                    "relation": relation,
                    "entity_side": entity_side,
                    "umls_search_ok": False,
                    "suggested_cui": pd.NA,
                    "suggested_umls_name": pd.NA,
                    "suggested_semantic_types": [],
                    "suggested_semantic_type_names": [],
                    "type_match_expected_category": pd.NA,
                }
            )
            continue

        cui, umls_name = best
        st_list, st_names = _umls_get_semantic_types(cui, UMLS_API_KEY)
        found = {str(x).strip().upper() for x in (st_list or []) if x}

        if not exp_tuis:
            match = pd.NA
        elif not found:
            match = pd.NA
        else:
            match = bool(found & exp_tuis)

        rows.append(
            {
                "original_name": orig,
                "suggested_name": sug,
                "expected_entity_type": exp_label,
                "expected_tuis": ",".join(sorted(exp_tuis)),
                "PMID": pmid,
                "relation": relation,
                "entity_side": entity_side,
                "umls_search_ok": True,
                "suggested_cui": cui,
                "suggested_umls_name": umls_name,
                "suggested_semantic_types": list(st_list or []),
                "suggested_semantic_type_names": list(st_names or []),
                "type_match_expected_category": match,
            }
        )

    return pd.DataFrame(rows)


suggested_umls_typecheck_df = build_suggested_umls_typecheck_df(suggested_name_replacement_df)


UMLS re-search suggested names: 100%|███████████████████████████████████████████████████████████████████████████| 838/838 [12:46<00:00,  1.09it/s]


In [ ]:
suggested_umls_typecheck_df

## Name-Filtering Pipeline and Stage Variables

This notebook filters curated entity names through PrimeKG and UMLS, then maps validated names back to relationship rows.

1) **Load and normalize curated relationships**
- `curated_relationships_df`: raw curated CSV input.
- `novel_relationships_df`: normalized relationship rows with `entity1`, `entity2`, `entity_type1`, `entity_type2`, `Relation`, and `PMID`.

2) **Build the initial query-name pool**
- `query_names`: unique names collected from both relationship sides.

3) **Run exact matching against PrimeKG**
- `kg_matched_names`: names that exact-match PrimeKG after normalization.
- `kg_nonmatched_names`: names that do not exact-match and require search-based handling.

4) **Attach curated entity-type context**
- `name_to_entity_types`: `name -> set(entity_type)` aggregated from both sides.
- `kg_matched_entities`, `kg_nonmatched_entities`: type context per bucket.

5) **Run UMLS normalization for nonmatched names**
- `entity_umls_df`: UMLS search output for nonmatched names (`matched_cui`, semantic types, etc.).
- Enriched metadata columns: `PMID`, `relation`, `entity_side`.

6) **Split by first-search CUI availability**
- `entity_with_cui_after_first_search`
- `entity_without_cui_after_first_search`

7) **Run semantic-type quality check on first-search CUI rows**
- Adds `expected_tuis` and `out_of_expected_tuis`.

8) **Build second-round suggestion inputs**
- `query_names` (round 2): quality-passed CUI names plus no-CUI names.

9) **Generate PrimeKG suggestions (SapBERT retrieve + SBERT rerank)**
- `suggested_name_replacement_df`:
  - `original_name`, `suggested_name`
  - merged metadata from `entity_umls_df`: `original_entity_types`, `PMID`, `relation`, `entity_side`.

10) **Type-check suggested names in UMLS**
- `suggested_umls_typecheck_df`:
  - re-search UMLS on `suggested_name`
  - compare expected TUIs vs found TUIs
  - output decision column: `type_match_expected_category`
  - carries curated metadata (`PMID`, `relation`, `entity_side`).

---

### Stage-wise name containers
- Initial pool: `query_names` (round 1)
- Exact matched: `kg_matched_names`
- Exact nonmatched: `kg_nonmatched_names`
- Nonmatched with UMLS context: `entity_umls_df`
- With first-search CUI: `entity_with_cui_after_first_search`
- Without first-search CUI: `entity_without_cui_after_first_search`
- Suggestion input pool: `query_names` (round 2, after quality filtering)
- Suggestion table: `suggested_name_replacement_df`
- Suggestion + type-check table: `suggested_umls_typecheck_df`

In [84]:
len(kg_matched_names)

246

In [85]:
valid_names_after_first_search = entity_with_cui_after_first_search[~entity_with_cui_after_first_search.out_of_expected_tuis].entity_name.unique()
len(valid_names_after_first_search)

242

In [ ]:
len(kg_matched_names)

In [ ]:
entity_with_cui_after_first_search[~entity_with_cui_after_first_search.out_of_expected_tuis]

In [ ]:
#second filtering logic
valid_names_after_first_search = entity_with_cui_after_first_search[~entity_with_cui_after_first_search.out_of_expected_tuis].entity_name.unique()
len(valid_names_after_first_search)

In [ ]:
suggested_umls_typecheck_df[suggested_umls_typecheck_df.type_match_expected_category==True]

## Final Consolidated Mapping Block

This block is the clean, production-ready version for mapping validated names back to relationship rows.

What it does:
- Builds `all_good_names` from exact KG matches + first-search valid names + second-search valid names.
- Builds one consolidated `name -> CUI` map from first and second UMLS searches.
- Creates `relationships_with_status_df` with:
  - `entity1_status`, `entity2_status`
  - values: `in_kg`, concrete CUI code(s) (`C...`), or `invalid`
- Adds second-search suggestion mapping:
  - `entity1_suggested_name`
  - `entity2_suggested_name`
  - `second_search_suggested_name` (combined view)

Notes:
- No debug `print` statements are used.
- Names not in `valid_names_after_second_search` get `None` in suggested-name columns.

In [96]:
# Consolidated final mapping (no debug prints)

# 1) Final valid-name sets
valid_names_after_first_search = entity_with_cui_after_first_search[
    entity_with_cui_after_first_search["out_of_expected_tuis"] == False
]["entity_name"].dropna().astype(str).str.strip().unique()

valid_names_after_second_search = suggested_umls_typecheck_df[
    suggested_umls_typecheck_df["type_match_expected_category"] == True
]["original_name"].dropna().astype(str).str.strip().unique()

all_good_names = sorted(set(kg_matched_names) | set(valid_names_after_first_search) | set(valid_names_after_second_search))

# 2) Build one name -> CUI map from first and second search results
name_to_cui = {}

first_valid_df = entity_with_cui_after_first_search[
    (entity_with_cui_after_first_search["out_of_expected_tuis"] == False)
    & (~entity_with_cui_after_first_search["matched_cui"].isna())
]
for _, r in first_valid_df.iterrows():
    n = _norm_name(r.get("entity_name"))
    c = str(r.get("matched_cui") or "").strip()
    if n and n != "nan" and c and c.lower() != "nan":
        name_to_cui.setdefault(n, set()).add(c)

second_valid_df = suggested_umls_typecheck_df[
    (suggested_umls_typecheck_df["type_match_expected_category"] == True)
    & (~suggested_umls_typecheck_df["suggested_cui"].isna())
]
for _, r in second_valid_df.iterrows():
    n = _norm_name(r.get("original_name"))
    c = str(r.get("suggested_cui") or "").strip()
    if n and n != "nan" and c and c.lower() != "nan":
        name_to_cui.setdefault(n, set()).add(c)

kg_matched_norm = {_norm_name(x) for x in kg_matched_names}


def _status_from_name(name: str) -> str:
    n = _norm_name(name)
    if not n or n == "nan":
        return "invalid"
    if n in kg_matched_norm:
        return "in_kg"
    cuis = sorted(name_to_cui.get(n, set()))
    if cuis:
        return "|".join(cuis)
    return "invalid"

# 3) Build status output mapped back to curated relationships
relationships_with_status_df = novel_relationships_df.copy()
relationships_with_status_df["entity1_status"] = relationships_with_status_df["entity1"].map(_status_from_name)
relationships_with_status_df["entity2_status"] = relationships_with_status_df["entity2"].map(_status_from_name)

# 4) Add second-search suggested names for original names only
valid_second_norm = {_norm_name(x) for x in valid_names_after_second_search}
second_search_map_df = suggested_name_replacement_df[
    suggested_name_replacement_df["original_name"].astype(str).str.strip().str.lower().isin(valid_second_norm)
][["original_name", "suggested_name"]].drop_duplicates(subset=["original_name"], keep="first")

second_search_name_to_suggested = {
    _norm_name(r["original_name"]): r["suggested_name"]
    for _, r in second_search_map_df.iterrows()
}

relationships_with_status_df["entity1_suggested_name"] = relationships_with_status_df["entity1"].map(
    lambda x: second_search_name_to_suggested.get(_norm_name(x), None)
)
relationships_with_status_df["entity2_suggested_name"] = relationships_with_status_df["entity2"].map(
    lambda x: second_search_name_to_suggested.get(_norm_name(x), None)
)


def _combined_suggestion(row):
    s1 = row.get("entity1_suggested_name")
    s2 = row.get("entity2_suggested_name")
    if pd.notna(s1) and pd.notna(s2):
        return f"entity1:{s1}|entity2:{s2}"
    if pd.notna(s1):
        return s1
    if pd.notna(s2):
        return s2
    return None

relationships_with_status_df["second_search_suggested_name"] = relationships_with_status_df.apply(
    _combined_suggestion,
    axis=1,
)

relationships_with_status_df.head(12)

,entity1,entity2,entity_type1,entity_type2,Relation,PMID,entity1_status,entity2_status,entity1_suggested_name,entity2_suggested_name,second_search_suggested_name
0,Niemann-Pick type C disease,NPC1,disease,protein/gene,disease_protein,34364974,invalid,in_kg,NaN,NaN,NaN
1,Niemann-Pick type C disease,NPC2,disease,protein/gene,disease_protein,34364974,invalid,in_kg,NaN,NaN,NaN
2,2-hydroxypropyl-beta-cyclodextrin,Niemann-Pick Disease Type C,drug,disease,indication,34364974,C0046237,in_kg,NaN,NaN,NaN
3,Niemann-Pick Disease Type C,neurovisceral symptoms,disease,phenotype,disease_phenotype_positive,34364974,in_kg,C1867971,NaN,Acute episodes of neuropathic symptoms,Acute episodes of neuropathic symptoms
4,Niemann-Pick Disease Type C,accumulation of cholesterol and other lipids,disease,phenotype,disease_phenotype_positive,34364974,in_kg,C5937001,NaN,Unesterified cholesterol accumulation in cultu...,Unesterified cholesterol accumulation in cultu...
5,2-hydroxypropyl-beta-cyclodextrin,release of lipid-laden lamellar inclusions,drug,biological_process,drug_bioprocess,34364974,C0046237,invalid,NaN,NaN,NaN
6,2-hydroxypropyl-beta-cyclodextrin,restore lipid homeostasis,drug,biological_process,drug_bioprocess,34364974,C0046237,C2262744,NaN,lipid homeostasis,lipid homeostasis
7,Niemann-Pick Disease Type C2,fatal neurovisceral disorder,disease,phenotype,disease_protein,37454976,invalid,invalid,NaN,NaN,NaN
8,Niemann-Pick Disease Type C2,NPC2,disease,protein/gene,disease_phenotype_positive,37454976,invalid,in_kg,NaN,NaN,NaN
9,Niemann-Pick Disease Type C2,splenomegaly,disease,phenotype,disease_phenotype_positive,37454976,invalid,in_kg,NaN,NaN,NaN


In [97]:
len(relationships_with_status_df)

2939

In [98]:
relationships_with_status_df[(relationships_with_status_df.entity1_status!="invalid")&(relationships_with_status_df.entity2_status!="invalid")]

,entity1,entity2,entity_type1,entity_type2,Relation,PMID,entity1_status,entity2_status,entity1_suggested_name,entity2_suggested_name,second_search_suggested_name
2,2-hydroxypropyl-beta-cyclodextrin,Niemann-Pick Disease Type C,drug,disease,indication,34364974,C0046237,in_kg,NaN,NaN,NaN
3,Niemann-Pick Disease Type C,neurovisceral symptoms,disease,phenotype,disease_phenotype_positive,34364974,in_kg,C1867971,NaN,Acute episodes of neuropathic symptoms,Acute episodes of neuropathic symptoms
4,Niemann-Pick Disease Type C,accumulation of cholesterol and other lipids,disease,phenotype,disease_phenotype_positive,34364974,in_kg,C5937001,NaN,Unesterified cholesterol accumulation in cultu...,Unesterified cholesterol accumulation in cultu...
6,2-hydroxypropyl-beta-cyclodextrin,restore lipid homeostasis,drug,biological_process,drug_bioprocess,34364974,C0046237,C2262744,NaN,lipid homeostasis,lipid homeostasis
20,lysosome,NPC2,cellular_component,protein/gene,cellcomp_protein,37454976,in_kg,in_kg,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2924,Niemann-Pick disease type C,NPC1,disease,protein/gene,disease_protein,40869913,in_kg,in_kg,NaN,NaN,NaN
2925,cerebral cortex,Niemann-Pick disease type C,anatomy,disease,anatomy_disease,40869913,in_kg,in_kg,NaN,NaN,NaN
2926,Niemann-Pick disease type C,lipid inclusions,disease,phenotype,disease_phenotype_positive,40869913,in_kg,C4231253,NaN,NaN,NaN
2927,Niemann-Pick disease type C,increased cholesterol,disease,phenotype,disease_phenotype_positive,40869913,in_kg,C3276464|C4551591,NaN,Increased HDL cholesterol concentration,Increased HDL cholesterol concentration


In [99]:
relationships_with_status_df[~relationships_with_status_df.second_search_suggested_name.isna()][["entity1", "entity2", "entity1_status", "entity2_status", "second_search_suggested_name"]]

In [100]:
final = relationships_with_status_df[(relationships_with_status_df.entity1_status!="invalid")&(relationships_with_status_df.entity2_status!="invalid")]

In [101]:
final.entity_type1.unique()

<StringArray>
[                  'drug',                'disease',     'cellular_component',
                'anatomy',              'phenotype',     'biological_process',
               'exposure',           'protein/gene',              'pathology',
               'chemical',                'pathway',     'molecular_function',
 'GRAS/Excipient/prodrug',                  'lipid']
Length: 14, dtype: str

In [102]:
final.entity_type2.unique()

<StringArray>
[           'disease',          'phenotype', 'biological_process',
       'protein/gene',            'anatomy',               'drug',
          'pathology', 'cellular_component', 'molecular_function',
            'pathway']
Length: 10, dtype: str

In [103]:
len(final)

729

In [104]:
final = final.dropna(subset=["entity_type1", "entity_type2"])
len(final)

729

In [ ]:
#Ask Thuy --> can we replace "Cell therapy" by "Drug"?
final.entity_type1.unique()

In [ ]:
final.entity_type2.unique()

In [105]:
final.to_csv(str(POST_DIR / "20260508-NMP_final.csv"))